# Project: Spectral clustering of the karate club 🥋
In 1977 a university karate club split in two after a conflict. From the friendship graph *alone*, can you predict who went with whom? The **Fiedler vector** (2nd eigenvector of the graph Laplacian) cuts the graph where it's thinnest, and recovers the real split almost perfectly.

Topic: [[Graph ML Overview]], [[Clustering]] · guide note: [[Project - Spectral Clustering of the Karate Club]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt, networkx as nx
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
G = nx.karate_club_graph()                                     # bundled with NetworkX (Zachary 1977)
A = nx.to_numpy_array(G, weight=None)
truth = np.array([0 if G.nodes[i]["club"] == "Mr. Hi" else 1 for i in G])
print(f"{G.number_of_nodes()} members, {G.number_of_edges()} friendships; split: {np.bincount(truth)}")

## 1. Normalised Laplacian\n$L = I - D^{-1/2}AD^{-1/2}$ with $D$ the degree matrix. Its eigenvalues lie in $[0, 2]$; the smallest is 0.

In [ ]:
def normalized_laplacian(A):
    # TODO 1: I − D^{-1/2} A D^{-1/2}
    raise NotImplementedError  # TODO 1

## 2. Fiedler cut
Take the eigenvector of the **second-smallest** eigenvalue (`np.linalg.eigh` sorts ascending) and assign each node to cluster 1 if its entry is positive, else 0.

In [ ]:
def fiedler_split(L):
    # TODO 2: sign of the second eigenvector
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_spec():
    two = np.zeros((6, 6))
    for a, b_ in [(0, 1), (1, 2), (0, 2), (3, 4), (4, 5), (3, 5), (2, 3)]: two[a, b_] = two[b_, a] = 1          # two triangles joined by one edge
    L = normalized_laplacian(two); s = fiedler_split(L)
    return np.allclose(L, L.T) and abs(np.linalg.eigvalsh(L)[0]) < 1e-9 and len(set(s[:3])) == 1 and len(set(s[3:])) == 1 and s[0] != s[3]
check("spectral", _t_spec, "Two triangles joined by one edge must be cut at that edge; L must be symmetric with smallest eigenvalue 0.")
if ready("spectral"):
    pred = fiedler_split(normalized_laplacian(A))
    acc = max(np.mean(pred == truth), np.mean(pred != truth))          # cluster labels are arbitrary
    print(f"Fiedler cut agrees with the real split for {acc:.0%} of members")
    wrong = [i for i in G if (pred[i] == truth[i]) != (np.mean(pred == truth) >= 0.5)]
    print("misassigned members:", wrong)
    pos = nx.spring_layout(G, seed=3); nx.draw(G, pos, node_color=pred, cmap="coolwarm", edgecolors=["k" if i in wrong else "w" for i in G], linewidths=2, with_labels=True, font_size=7); plt.show()
    check("recovers the split", lambda: acc >= 0.9, "The Fiedler cut should match the real split for ≥ 90 % of members.")

<details><summary>▶ Solution</summary>

```python
def normalized_laplacian(A):
    d = A.sum(1)
    Dm = np.diag(1 / np.sqrt(d))
    return np.eye(len(A)) - Dm @ A @ Dm
```

```python
def fiedler_split(L):
    vals, vecs = np.linalg.eigh(L)
    return (vecs[:, 1] > 0).astype(int)
```
</details>

## Stretch goals
- k > 2 clusters: k-means on the first k eigenvectors ([[Clustering]]).
- Compare with modularity-based community detection (`nx.community.greedy_modularity_communities`).
- Run it on your vault's link graph from [[Project - PageRank of My Vault]]: do the clusters match your folders?